# G3-VPR v2: Notebook Huấn Luyện SOTA & Ablation Tự Động 100% Trên Kaggle (`T4 x2`)
**Bản nâng cấp v2 đã tích hợp:**
- **Isometric Multi-Scale Orthogonal Matryoshka Head** & **Symmetric Signed GeM** (bảo toàn 100% không gian đặc trưng).
- **MatryoshkaTeacherBridge (`MegaLoc` 8448-D Direct Subspace) + Negative-Masked Geodesic Relational Distillation**.
- **Khởi động ấm thân mạng `ViT-S + MulConvAdapter`** tự động tải từ GitHub + **Bảng Ablation `M1 -> M5_full` thuần Python**.

**Cách chạy trên Kaggle:**
1. Ở **Settings** bên phải: chọn **Accelerator -> `GPU T4 x2`** và bật **Internet -> `ON`**.
2. Ở **Input** bên phải: bấm **`+ Add Input`** -> gắn **`amaralibey/gsv-cities`**.
3. Bấm **`Save Version` -> `Save & Run All (Commit)`**!

In [ ]:
# Cell 1: Tu dong giai nen toan bo code G3-VPR v2 ngay trong Kaggle/Colab
import base64, io, os, subprocess, sys, tarfile, torch

WORK_ROOT = "/kaggle/working/D2VPR" if os.path.isdir("/kaggle/working") else os.path.abspath("./D2VPR_cloud")
os.makedirs(WORK_ROOT, exist_ok=True)

ARCHIVE_B64 = (
    "..."
)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ARCHIVE_B64)), mode="r:gz") as tf:
    tf.extractall(WORK_ROOT)
os.chdir(WORK_ROOT)
print("[OK] Da giai nen toan bo du an G3-VPR v2 tai:", os.getcwd())
print("PyTorch:", torch.__version__, "| CUDA GPU:", torch.cuda.is_available(), "| So GPU:", torch.cuda.device_count())
if not torch.cuda.is_available():
    print("CANH BAO: Ban chua bat GPU! Vao Settings ben phai -> Accelerator -> chon GPU T4 x2 va bat Internet ON!")
else:
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pytorch-metric-learning"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)
print("[OK] Moi truong thu vien da san sang 100%!")

In [ ]:
# Cell 2: Tai trong so khoi dong am ViT-S+Adapter (v2), DINOv2-S & Tu dong tim GSV-Cities
import os, urllib.request
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

os.makedirs("./weights_init", exist_ok=True)
warm_ckpt = "./weights_init/vits_adapter_warmstart_fp16.pt"
if not os.path.isfile(warm_ckpt):
    warm_urls = [
        "https://raw.githubusercontent.com/phuccode25/G3-VPR/main/weights_init/vits_adapter_warmstart_fp16.pt",
        "https://github.com/phuccode25/G3-VPR/raw/main/weights_init/vits_adapter_warmstart_fp16.pt",
    ]
    for u in warm_urls:
        try:
            print("Dang tai trong so khoi dong am ViT-S+Adapter tu:", u)
            urllib.request.urlretrieve(u, warm_ckpt)
            if os.path.isfile(warm_ckpt) and os.path.getsize(warm_ckpt) > 10_000_000:
                break
        except Exception as e:
            print("Thu URL tiep theo do:", e)
print("[OK] Warm-start ViT-S san sang:", os.path.isfile(warm_ckpt), f"({os.path.getsize(warm_ckpt)/1e6:.1f} MB)" if os.path.isfile(warm_ckpt) else "")

dinov2_ckpt = "./dinov2_vits14_pretrain.pth"
if not os.path.isfile(dinov2_ckpt):
    print("Dang tai trong so DINOv2 ViT-S/14...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth",
        dinov2_ckpt
    )
print("[OK] DINOv2-S san sang:", os.path.isfile(dinov2_ckpt))

GSV_PATH = None
if os.path.isdir("/kaggle/input"):
    for r, dirs, _ in os.walk("/kaggle/input"):
        if "Dataframes" in dirs and "Images" in dirs:
            GSV_PATH = r
            break
if GSV_PATH is None:
    import kagglehub
    GSV_PATH = kagglehub.dataset_download("amaralibey/gsv-cities")
print("[OK] Da tim thay GSV-Cities tai:", GSV_PATH)

In [ ]:
# Cell 3: Kiem tra nhanh kien truc G3-VPR v2, tinh bat bien BS=1 & toc do GPU
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")
subprocess.run([sys.executable, "-u", "experiments/exp2_architecture_verify_and_profile.py"], check=True)

In [ ]:
# Cell 4: Huan luyen Mo hinh Chinh G3-VPR v2 (M5_full) voi Dual Teachers (MegaLoc 2D + Depth-Anything-V2 3D)
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

SAVE_DIR = "./checkpoints/g3vpr_M5_full"
cmd = [
    sys.executable, "-u", "train_g3vpr.py",
    "--ablation", "M5_full",
    "--gsv_path", GSV_PATH,
    "--dinov2_path", "./dinov2_vits14_pretrain.pth",
    "--teacher_2d", "megaloc",
    "--teacher_3d", "depth_anything_v2",
    "--bs", "32",
    "--grad_accum", "2",
    "--epochs", "8",
    "--max_places", "22000",
    "--lr", "3e-4",
    "--unfreeze_last_n", "6",
    "--save_dir", SAVE_DIR,
]
if os.path.isfile("./weights_init/vits_adapter_warmstart_fp16.pt"):
    cmd.extend(["--warmstart_ckpt", "./weights_init/vits_adapter_warmstart_fp16.pt"])
if os.path.isfile(f"{SAVE_DIR}/g3vpr_last.pth"):
    cmd.extend(["--resume_ckpt", f"{SAVE_DIR}/g3vpr_last.pth"])

print("Running M5_full:", " ".join(cmd))
subprocess.run(cmd, stdin=subprocess.DEVNULL, check=True)
# Xoa file last.pth nang de tiet kiem o dia cho Cell 5
if os.path.isfile(f"{SAVE_DIR}/g3vpr_last.pth"):
    os.remove(f"{SAVE_DIR}/g3vpr_last.pth")

In [ ]:
# Cell 5: Chay Bang Ablation Study Thuan Python 100% (Table 4: M1 -> M2 -> M3 -> M4)
import os, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

for abl in ["M1", "M2", "M3", "M4"]:
    out_d = f"./checkpoints/ablation_{abl}"
    use_distill = (abl == "M4")
    abl_cmd = [
        sys.executable, "-u", "train_g3vpr.py",
        "--ablation", abl,
        "--gsv_path", GSV_PATH,
        "--cities", "all",
        "--dinov2_path", "./dinov2_vits14_pretrain.pth",
        "--teacher_2d", "megaloc" if use_distill else "none",
        "--teacher_3d", "depth_anything_v2" if use_distill else "none",
        "--bs", "32",
        "--grad_accum", "2",
        "--epochs", "4",
        "--max_places", "6000",
        "--lr", "3e-4",
        "--unfreeze_last_n", "6",
        "--save_dir", out_d,
    ]
    if os.path.isfile("./weights_init/vits_adapter_warmstart_fp16.pt"):
        abl_cmd.extend(["--warmstart_ckpt", "./weights_init/vits_adapter_warmstart_fp16.pt"])
    print(f"\n=== Running Ablation {abl} ===")
    subprocess.run(abl_cmd, stdin=subprocess.DEVNULL, check=True)
    # Giu lai g3vpr_best_fp16.pth (~68 MB) + training_metrics.json, dọn file fp32 trung gian
    for rm_fn in ("g3vpr_last.pth", "g3vpr_universal_best.pth"):
        p_rm = os.path.join(out_d, rm_fn)
        if os.path.isfile(p_rm):
            os.remove(p_rm)

In [ ]:
# Cell 6: Danh gia BS=1 Matryoshka (128D..4096D), ve Hinh 4 & Dong goi ZIP ket qua v2!
import os, shutil, subprocess, sys
if os.path.isdir("/kaggle/working/D2VPR"):
    os.chdir("/kaggle/working/D2VPR")

m5_ckpt = "./checkpoints/g3vpr_M5_full/g3vpr_universal_best.pth"
if os.path.isfile(m5_ckpt):
    subprocess.run([
        sys.executable, "-u", "eval_g3vpr_sota.py",
        "--ckpt", m5_ckpt,
        "--query_bs", "1",
        "--output_json", "./results/g3vpr_sota_eval.json",
    ], check=True)
    subprocess.run([
        sys.executable, "-u", "experiments/visualize_gaussians.py",
        "--ckpt", m5_ckpt,
    ], check=True)

zip_out = "/kaggle/working/g3vpr_paper_artifacts_v2" if os.path.isdir("/kaggle/working") else "./g3vpr_paper_artifacts_v2"
shutil.rmtree("./export_bundle", ignore_errors=True)
os.makedirs("./export_bundle/results", exist_ok=True)
os.makedirs("./export_bundle/checkpoints/g3vpr_M5_full", exist_ok=True)
if os.path.isdir("./results"):
    shutil.copytree("./results", "./export_bundle/results", dirs_exist_ok=True)

for main_fn in ("g3vpr_universal_best.pth", "g3vpr_best_fp16.pth", "training_metrics.json"):
    src_p = f"./checkpoints/g3vpr_M5_full/{main_fn}"
    if os.path.isfile(src_p):
        shutil.copy2(src_p, f"./export_bundle/checkpoints/g3vpr_M5_full/{main_fn}")

# Sao chep day du trong so fp16 + JSON cua M1..M4 de cham SOTA tren SPED/AmsterTime/Pitts30k o may local!
for abl in ("M1", "M2", "M3", "M4"):
    abl_src = f"./checkpoints/ablation_{abl}"
    if os.path.isdir(abl_src):
        abl_dst = f"./export_bundle/checkpoints/ablation_{abl}"
        os.makedirs(abl_dst, exist_ok=True)
        for fn in ("g3vpr_best_fp16.pth", "training_metrics.json"):
            fp = os.path.join(abl_src, fn)
            if os.path.isfile(fp):
                shutil.copy2(fp, os.path.join(abl_dst, fn))

archive_file = shutil.make_archive(zip_out, "zip", "./export_bundle")
print("[HOAN TAT V2] Tai file ket qua tai muc Output ben phai:", archive_file)